# Memory Management

Agent memory stores useful information outside the current prompt and retrieves it when needed later.


## What to learn

- Working memory is the information currently in context.
- Session memory summarizes earlier parts of the same task.
- Long-term memory stores durable facts across sessions.
- Retrieval selects memories relevant to the current step.

## Decision rule

Store information only when it is useful, durable, and permitted. Keep its source and update time, retrieve narrowly, and provide deletion and correction paths for personal data.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Context-budget management with structured compaction, plus a
recency-aware memory retriever with correction handling.
"""

# --- 1. compaction under a token budget -----------------------------------
BUDGET = 100  # toy token budget (prod: model window minus output headroom)
est = lambda text: len(text.split())  # toy tokenizer


# Define the data shape and small operations before running them.
def compact(turns):
    """Structured compaction: preserve decisions/constraints/ids verbatim."""
    summary = {"decisions": [], "constraints": [], "state": ""}
    for t in turns:
        if "decided" in t:
            summary["decisions"].append(t)          # verbatim, not paraphrased
        elif "must" in t or "never" in t:
            summary["constraints"].append(t)
    summary["state"] = f"({len(turns)} earlier turns compacted)"
    return summary


history = [
    "user: we decided to use Postgres for the transcript store",
    "assistant: noted, setting that up",
    "user: the demo must ship Friday",
    "assistant: understood",
    "user: how should we shard the vector index?",
    "assistant: for 250k vectors pgvector needs no sharding at all",
    "user: ok what about the dashboard queries",
]

used = sum(est(t) for t in history)
if used > BUDGET * 0.7:  # compact at 70%, not at overflow
    keep_recent = history[-2:]
    summary = compact(history[:-2])
    print(f"compacted: {used} -> ~{est(str(summary)) + sum(est(t) for t in keep_recent)} tokens")
    print(" preserved decisions:", summary["decisions"])
    print(" preserved constraints:", summary["constraints"])

# --- 2. retrieval-based memory with corrections ----------------------------
MEMORIES = [
    {"id": 1, "text": "user prefers formal tone in emails", "age_days": 90, "superseded_by": 3},
    {"id": 2, "text": "user's company is AcmeCo, fiscal year ends March", "age_days": 40, "superseded_by": None},
    {"id": 3, "text": "user now prefers casual tone in emails", "age_days": 5, "superseded_by": None},
]


def retrieve(query_terms, k=2):
    """similarity x recency, excluding tombstoned entries."""
    live = [m for m in MEMORIES if m["superseded_by"] is None]
    def score(m):
        sim = len(set(query_terms) & set(m["text"].split())) / len(query_terms)
        recency = 1.0 / (1 + m["age_days"] / 30)
        return sim * recency
    return sorted(live, key=score, reverse=True)[:k]


hits = retrieve(["tone", "emails", "draft"])
print("\nretrieved for 'draft an email':")
for m in hits:
    print(f"  #{m['id']}: {m['text']}")
# The stale 'formal tone' memory is tombstoned - only the correction returns.


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
